In [1]:
import numpy as np
import pandas as pd
import os
import gc
import datetime as dt
import warnings
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, matthews_corrcoef)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, BatchNormalization, Dense, Dropout
from tensorflow.keras.callbacks import CSVLogger
from tensorflow.keras import mixed_precision

# --- 1. HARDWARE INITIALIZATION ---
warnings.filterwarnings('ignore')
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    try:
        # Enable Mixed Precision (FP16) for RTX 4070
        mixed_precision.set_global_policy('mixed_float16')
        # Enable XLA Compiler
        tf.config.optimizer.set_jit(True)
        print(f"[{dt.datetime.now().time()}] SUCCESS: GPU {gpus[0]} detected. Acceleration Active.")
    except Exception as e:
        print(f"Hardware initialization failed: {e}")
else:
    print("!!! WARNING: GPU NOT DETECTED. Check LD_LIBRARY_PATH.")

# --- 2. CONFIGURATION ---
DATASETS = [
    "~/Downloads/Dataset/BASE_PAPER.csv",
    "~/Downloads/Dataset/CIC-IDS- 2017/Monday-WorkingHours.pcap_ISCX.csv",
    "~/Downloads/Dataset/CIC-IDS- 2017/Tuesday-WorkingHours.pcap_ISCX.csv",
    "~/Downloads/Dataset/CIC-IDS- 2017/Wednesday-workingHours.pcap_ISCX.csv",
    "~/Downloads/Dataset/CIC-IDS- 2017/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "~/Downloads/Dataset/CIC-IDS- 2017/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
    "~/Downloads/Dataset/CIC-IDS- 2017/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "~/Downloads/Dataset/CSE-CIC-IDS2018/02-14-2018.csv",
    "~/Downloads/Dataset/CSE-CIC-IDS2018/02-21-2018.csv",
    "~/Downloads/Dataset/CSE-CIC-IDS2018/03-01-2018.csv",
]
BATCH_SIZE = 8192 # Sweet spot for 12GB VRAM
benchmark_results = []

def create_model(input_shape, num_classes):
    model = Sequential([
        Conv1D(64, 5, activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(3, strides=2, padding='same'),
        Conv1D(128, 5, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(3, strides=2, padding='same'),
        Conv1D(256, 5, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(3, strides=2, padding='same'),
        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.3),
        Dense(128, activation='relu'),
        Dense(num_classes, activation='softmax', dtype='float32') # Float32 for stability
    ])
    model.compile(
        loss='categorical_crossentropy', 
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
        metrics=['accuracy'],
        jit_compile=True # Speed boost
    )
    return model

def process_dataset(file_path):
    dataset_name = os.path.basename(file_path)
    full_path = os.path.expanduser(file_path)
    
    print(f"[{dt.datetime.now().time()}] Loading: {dataset_name}")
    try:
        df = pd.read_csv(full_path, encoding='utf-8', low_memory=False)
    except:
        df = pd.read_csv(full_path, encoding='cp1252', low_memory=False)

    df.columns = df.columns.str.strip()
    target_col = next((col for col in df.columns if 'label' in col.lower()), None)
    if not target_col: return None
    df.rename(columns={target_col: 'Label'}, inplace=True)
    
    # Cleaning
    X_raw = df.drop(columns=['Label', 'Timestamp'], errors='ignore').apply(pd.to_numeric, errors='coerce')
    data_clean = pd.concat([X_raw, df['Label']], axis=1).replace([np.inf, -np.inf], np.nan).dropna()
    
    X = StandardScaler().fit_transform(data_clean.drop(columns=['Label']).values)
    le = LabelEncoder()
    Y = OneHotEncoder(sparse_output=False).fit_transform(le.fit_transform(data_clean['Label']).reshape(-1, 1))

    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
    X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
    X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

    model = create_model((X_train.shape[1], 1), len(le.classes_))
    
    print(f"   -> Training (Batch: {BATCH_SIZE})...")
    start = dt.datetime.now()
    model.fit(X_train, Y_train, epochs=10, batch_size=BATCH_SIZE, validation_data=(X_test, Y_test), verbose=1)
    duration = dt.datetime.now() - start

    pred = np.argmax(model.predict(X_test, batch_size=BATCH_SIZE), axis=1)
    y_true = np.argmax(Y_test, axis=1)

    # Cleanup
    del df, data_clean, X_train, X_test, model
    tf.keras.backend.clear_session()
    gc.collect()

    return {"Dataset": dataset_name, "Accuracy": accuracy_score(y_true, pred), "Duration": str(duration)}

# --- 3. MAIN EXECUTION ---
print("\n" + "="*40 + "\nSTARTING BENCHMARK\n" + "="*40)
for path in DATASETS:
    if os.path.exists(os.path.expanduser(path)):
        try:
            res = process_dataset(path)
            if res: benchmark_results.append(res)
        except Exception as e:
            print(f"Error processing {path}: {e}")
    else:
        print(f"[MISSING] File not found: {path}")

if benchmark_results:
    report = pd.DataFrame(benchmark_results)
    print("\nFINAL REPORT:\n", report)
    report.to_csv("Benchmark_Results.csv", index=False)

2025-12-24 12:16:14.307733: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-24 12:16:14.307961: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-24 12:16:14.339592: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-24 12:16:15.147846: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

!!! WARNING: GPU NOT DETECTED. Check LD_LIBRARY_PATH.

STARTING BENCHMARK
[12:16:15.621422] Loading: BASE_PAPER.csv


E0000 00:00:1766558775.615700   38431 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1766558775.620185   38431 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


   -> Training (Batch: 8192)...
Epoch 1/10


2025-12-24 12:16:17.289743: I external/local_xla/xla/service/service.cc:163] XLA service 0x7ebd14013260 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2025-12-24 12:16:17.289759: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): Host, Default Version
2025-12-24 12:16:17.319807: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1766558777.855277   42034 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


11/11 ━━━━━━━━━━━━━━━━━━━━ 12s 969ms/step - accuracy: 0.6788 - loss: 1.1131 - val_accuracy: 0.3794 - val_loss: 2.4808
Epoch 2/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 10s 892ms/step - accuracy: 0.8487 - loss: 0.3795 - val_accuracy: 0.3828 - val_loss: 2.4108
Epoch 3/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 10s 889ms/step - accuracy: 0.8701 - loss: 0.2940 - val_accuracy: 0.2951 - val_loss: 2.4022
Epoch 4/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 10s 959ms/step - accuracy: 0.8787 - loss: 0.2611 - val_accuracy: 0.2223 - val_loss: 2.4159
Epoch 5/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 10s 955ms/step - accuracy: 0.8840 - loss: 0.2422 - val_accuracy: 0.1885 - val_loss: 2.4788
Epoch 6/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 10s 957ms/step - accuracy: 0.8881 - loss: 0.2326 - val_accuracy: 0.1650 - val_loss: 2.5993
Epoch 7/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 11s 958ms/step - accuracy: 0.8897 - loss: 0.2272 - val_accuracy: 0.1676 - val_loss: 2.7402
Epoch 8/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 11s 958ms/step - accuracy: 0.8913 - loss: 0.2218 - val_accuracy: 0.137